# Matched geometry diagnostic — review only
Private saved batch; two T4s, original two-hour deadline. No expansion/refit/DFT.

In [ ]:
# Preparation template only. Enable solely after explicit diagnostic approval.
APPROVED = False
assert APPROVED, 'Not authorized: review the diagnostic and two-T4, two-hour cap first'
import os, sys, time, json, subprocess, zipfile, hashlib, traceback
from pathlib import Path
from kaggle_secrets import UserSecretsClient
W = Path('/kaggle/working')
receipt = W/'matched_allocation_start_v2.json'
assert not receipt.exists(), 'Never refresh an allocation or silently rerun'
START = time.time()
receipt.write_text(json.dumps({'started_epoch': START, 'deadline_epoch': START+7200}))
ROOT = W/'matched_diagnostic_frozen_v2'
OUT = W/'matched_diagnostic_results_v2'
def remaining():
    value = START+6900-time.time()
    if value <= 0: raise TimeoutError('Original allocation exhausted')
    return value
try:
    gpu = subprocess.check_output(['nvidia-smi','--query-gpu=index,name,uuid,memory.total','--format=csv,noheader'],text=True,timeout=remaining())
    (W/'matched_gpu_observation_v2.txt').write_text(gpu)
    assert len(gpu.strip().splitlines()) == 2 and all('T4' in x for x in gpu.splitlines())
    bundles = list(Path('/kaggle/input').rglob('UIO66_Matched_Diagnostic_v2.zip'))
    assert len(bundles) == 1 and not ROOT.exists()
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist(): assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
        z.extractall(ROOT)
    release = json.loads((ROOT/'release.json').read_text())
    for name,digest in release['sha256'].items():
        p = ROOT/name
        assert hashlib.sha256(p.read_bytes()).hexdigest() == digest, name
    os.chdir(ROOT)
    env = dict(os.environ, PYTHONPATH=str(ROOT), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONDONTWRITEBYTECODE='1')
    with (W/'matched_install_v2.log').open('w') as log:
        subprocess.run([sys.executable,'-m','pip','install','fairchem-core==2.23.0','ase==3.26.0'],stdout=log,stderr=subprocess.STDOUT,check=True,timeout=remaining())
    subprocess.run([sys.executable,'-m','scripts.run_uio66_matched_diagnostic','--manifest',str(ROOT/'manifest.json')],env=env,check=True,timeout=remaining())
    env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    download = "from huggingface_hub import hf_hub_download; from pathlib import Path; import json; p=json.loads(Path('manifest.json').read_text()); q=hf_hub_download('facebook/UMA','checkpoints/uma-s-1p2p1.pt',revision=p['checkpoint_revision']); Path('/kaggle/working/matched_checkpoint_path_v2.txt').write_text(q)"
    subprocess.run([sys.executable,'-c',download],env=env,check=True,timeout=remaining())
    checkpoint = (W/'matched_checkpoint_path_v2.txt').read_text()
    env.pop('HF_TOKEN',None)
    command = [sys.executable,'-m','scripts.run_uio66_matched_diagnostic','--manifest',str(ROOT/'manifest.json'),
               '--output',str(OUT),'--checkpoint',checkpoint,'--execute','--approved-wall-hours','2',
               '--allocation-start-epoch',str(START)]
    # Controller owns the hard deadline and terminates both model workers.
    subprocess.run(command,env=env,check=True)
    subprocess.run([sys.executable,'-m','scripts.assess_uio66_matched_diagnostic','--manifest',str(ROOT/'manifest.json'),'--output',str(OUT)],env=env,check=True,timeout=max(1,START+7050-time.time()))
except BaseException:
    (W/'matched_workflow_error_v2.txt').write_text(traceback.format_exc())
    raise
finally:
    # Saved private batch output: retain setup failures and partial raw trajectories too.
    with zipfile.ZipFile(W/'matched_diagnostic_evidence_v2.zip','w',zipfile.ZIP_STORED) as z:
        for p in W.rglob('*'):
            if p.is_file() and p.suffix != '.zip' and (p.is_relative_to(OUT) or p.parent == W or p.is_relative_to(ROOT)):
                z.write(p,p.relative_to(W))
